In [1]:
import importlib
from io import StringIO
from typing import cast
import psycopg
from psycopg.abc import Params, Query
import matplotlib.pyplot as plt
from matplotlib import style, rcParams
from pandas_datareader import wb
import pandas as pd
import statsmodels.formula.api as smf
from src.utils import apply_matplotlib_settings
from src.worldbank.utils import db

apply_matplotlib_settings()

In [2]:
gdp_matches = wb.search("gdp.*capita.*const").iloc[:, :2]
with pd.option_context("display.max_colwidth", None):
    display(gdp_matches)

,id,name
691,6.0.GDPpc_constant,"GDP per capita, PPP (constant 2011 international $)"
11160,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$)
11162,NY.GDP.PCAP.KN,GDP per capita (constant LCU)
11164,NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 international $)"
11165,NY.GDP.PCAP.PP.KD.87,"GDP per capita, PPP (constant 1987 international $)"


In [3]:
indicator = gdp_matches.loc[11160]["id"]
gdp = {
    "raw": wb.download(
        indicator=indicator, country="all", start=2010, end=2023
    )
}
gdp["raw"]

/tmp/ipykernel_1529/3239662871.py:3: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  "raw": wb.download(


NY.GDP.PCAP.KD
country                     year                
Africa Eastern and Southern 2023     1418.363737
                            2022     1421.589728
                            2021     1408.860838
                            2020     1383.377818
                            2019     1463.139549
...                                          ...
Zimbabwe                    2014     1377.254560
                            2013     1375.851342
                            2012     1352.135136
                            2011     1187.318407
                            2010     1054.397915

[3724 rows x 1 columns]

In [4]:
gdp["renamed"] = gdp["raw"].rename(columns={(gdp["raw"].columns[0]): "value"})
gdp["dropna"] = gdp["renamed"].dropna()
gdp["dropna"].columns[0]
gdp["types"] = (
    gdp["dropna"]
    .reset_index()
    .astype({"country": "string", "year": "int", "value": "float"})
)

gdp["types"]

,country,year,value
0,Africa Eastern and Southern,2023,1418.363737
1,Africa Eastern and Southern,2022,1421.589728
2,Africa Eastern and Southern,2021,1408.860838
3,Africa Eastern and Southern,2020,1383.377818
4,Africa Eastern and Southern,2019,1463.139549
...,...,...,...
3571,Zimbabwe,2014,1377.254560
3572,Zimbabwe,2013,1375.851342
3573,Zimbabwe,2012,1352.135136
3574,Zimbabwe,2011,1187.318407


In [5]:
from src.worldbank.utils import db

importlib.reload(db)
sql = db.get_queries_by_type("gdp")

In [6]:
rows = [tuple(p) for p in gdp["types"].to_numpy()]
with db.postgres_context() as (conn, cur):
    cur.execute(sql["table.create"]["sql"])
    cur.executemany(sql["insert.one"]["sql"], rows)
    cur.execute(sql["select.all"]["sql"])
    response = cur.fetchall()
    print(len(response))

3576


In [8]:
columns = db.select("columns.json", ["gdp"])[0][0].values()
columns

dict_values(['id', 'country', 'year', 'value'])

In [11]:
df = pd.DataFrame(db.select("all"), columns=columns).set_index("id")
df

,country,year,value
id,,,
1,Africa Eastern and Southern,2023,1418.363737
2,Africa Eastern and Southern,2022,1421.589728
3,Africa Eastern and Southern,2021,1408.860838
4,Africa Eastern and Southern,2020,1383.377818
5,Africa Eastern and Southern,2019,1463.139549
...,...,...,...
3572,Zimbabwe,2014,1377.254560
3573,Zimbabwe,2013,1375.851342
3574,Zimbabwe,2012,1352.135136


In [12]:
df = pd.DataFrame(db.select("country.max-and-diff"), columns=columns)
df

,id,country,year,value
0,Afghanistan,2012,568.929021,48.214095
1,Afghanistan,2013,580.603833,59.888907
2,Afghanistan,2014,575.146246,54.431319
3,Afghanistan,2015,565.569730,44.854804
4,Afghanistan,2016,563.872337,43.157410
...,...,...,...,...
3571,Zimbabwe,2018,1464.315294,132.520344
3572,Zimbabwe,2019,1350.309851,18.514901
3573,Zimbabwe,2020,1224.272314,-107.522635
3574,Zimbabwe,2021,1305.220113,-26.574837


In [13]:
import importlib
from src.worldbank.utils import db

importlib.reload(db)

df = pd.DataFrame(
    db.select("country.max-and-year"), columns=columns
).set_index("id")

with pd.option_context("display.max_rows", 100):
    display(df)

,country,year,value
id,,,
2414,Monaco,2023,224582.45
2203,Liechtenstein,2015,167187.16
979,Bermuda,2010,118382.91
2220,Luxembourg,2021,110425.89
1927,Ireland,2022,99677.47
...,...,...,...
1186,Central African Republic,2012,553.73
1273,"Congo, Dem. Rep.",2023,537.27
3049,Somalia,2017,506.50


In [14]:
df = pd.DataFrame(db.select("country.max-value-last-year"), columns=columns)
df

,id,country,year,value
0,683,Afghanistan,2013,580.603833
1,674,Afghanistan,2022,377.665627
2,685,Afghanistan,2011,525.426983
3,681,Afghanistan,2015,565.569730
4,675,Afghanistan,2021,408.625855
...,...,...,...,...
1334,3563,Zimbabwe,2023,1410.737311
1335,3575,Zimbabwe,2011,1187.318407
1336,3564,Zimbabwe,2022,1361.914530
1337,3568,Zimbabwe,2018,1464.315294


In [15]:
db.exec("function.latest-year")
df = pd.DataFrame(db.select("country.max-year"), columns=columns)
df

,id,country,year,value
0,673,Afghanistan,2023,379.707497
1,1,Africa Eastern and Southern,2023,1418.363737
2,15,Africa Western and Central,2023,1820.754741
3,687,Albania,2023,5419.637791
4,701,Algeria,2023,4660.405457
...,...,...,...,...
255,3521,West Bank and Gaza,2023,2862.881175
256,659,World,2023,11578.780010
257,3535,"Yemen, Rep.",2023,859.209303
258,3549,Zambia,2023,1330.859982


In [7]:
db.exec("view.latest-year")
df = pd.DataFrame(db.select("country.view.latest-year"))
df

,0,1,2,3
0,5141,Afghanistan,2023,379.707497
1,4469,Africa Eastern and Southern,2023,1418.363737
2,4483,Africa Western and Central,2023,1820.754741
3,5155,Albania,2023,5419.637791
4,5169,Algeria,2023,4660.405457
...,...,...,...,...
255,7989,West Bank and Gaza,2023,2862.881175
256,5127,World,2023,11578.780010
257,8003,"Yemen, Rep.",2023,859.209303
258,8017,Zambia,2023,1330.859982


In [16]:
db.exec("view.country.last-and-max")
df = pd.DataFrame(db.select("view.country.last-and-max"), columns=columns)
df

,id,country,year,value
0,683,Afghanistan,2013,580.603833
1,673,Afghanistan,2023,379.707497
2,9,Africa Eastern and Southern,2015,1479.564123
3,1,Africa Eastern and Southern,2023,1418.363737
4,24,Africa Western and Central,2014,1846.050944
...,...,...,...,...
372,3548,"Yemen, Rep.",2010,2356.199574
373,3535,"Yemen, Rep.",2023,859.209303
374,3549,Zambia,2023,1330.859982
375,3568,Zimbabwe,2018,1464.315294
